In [ ]:
# Setup: check GPU, clone or pull logseer repo and import core libraries
import subprocess, os, sys
import tensorflow as tf
import numpy as np
import pickle
import xgboost as xgb
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.callbacks import ModelCheckpoint, EarlyStopping
from keras.models import load_model
from keras import optimizers
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import binomtest

if not tf.config.list_physical_devices('GPU'):
    print("⚠️ WARNING: No GPU detected! Will run on CPU.")
else:
    print("✅ GPU is available:", tf.config.list_physical_devices('GPU'))
if not os.path.exists('../logseer'):
    !git clone https://github.com/masahiko-shibata/logseer.git ../logseer
else:
    !cd ../logseer && git pull
sys.path.insert(0, '../logseer')
from logseer import *
import keras

os.chdir('..')

In [ ]:
# Run this to load log data from Google Drive to Colab. Otherwise, copy the data folder to DATA_DIR.
from google.colab import drive
import shutil, zipfile
drive.mount('/content/drive')

!rm -rf logs data.zip 2>/dev/null
shutil.copy('/content/drive/MyDrive/Colab Notebooks/logseer/data/data.zip', 'data.zip')
with zipfile.ZipFile('data.zip', 'r') as z:
    z.extractall('.')
print(os.listdir('./'))

In [ ]:
# Configuration

# Paths
BASE_DIR            = '../'
DATA_DIR            = 'logs'
MODEL_SAVE_PATH     = 'logseer.keras'
TOKENIZER_PATH      = 'tokenizer.pickle'

# Data
NUM_CHAR            = 3000
TO_ID               = 6000
VALIDATION_SPLIT      = 0.1
VALIDATE_ON_TEST_DATA = False
SUCCESS_LOG_RATIO     = 99

# Model
MODEL_NAME          = 'LogCNNv2'
EMBEDDING_LAYER     = 'vanilla'
EMBEDDING_DIM       = 32
MAX_NB_WORDS        = 24000
MAX_SEQUENCE_LENGTH = 26000

# Training
EPOCHS              = 60
BATCH_SIZE          = 32
LEARNING_RATE       = 0.0003
REPETITION          = 100
RETRAIN             = False
NN_ERROR_WEIGHT     = 2
LABEL_SMOOTHING     = 0.0

# Early stopping
USE_EARLY_STOPPING   = True
PATIENCE             = 15
RESTORE_BEST_WEIGHTS = False
ES_START_FROM_EPOCH  = 10
MONITOR              = 'val_f1'
MODE                 = 'max'

# Checkpoint: 'multi_metric' | 'best_f1' | 'standard'
CHECKPOINT_TYPE      = 'best_f1'
START_FROM_EPOCH     = 0
MAX_LOSS            = 0.7

# Sklearn models to train alongside the NN: 'xgb', 'svm', 'rf' (comma-separated string or list)
SKLEARN_MODELS      = 'xgb'

In [ ]:
# Training loop
tester = run_training(
    DATA_DIR,
    max_nb_words=MAX_NB_WORDS,
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    embedding_dim=EMBEDDING_DIM,
    validation_split=VALIDATION_SPLIT,
    validate_on_test_data=VALIDATE_ON_TEST_DATA,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    model_save_path=MODEL_SAVE_PATH,
    tokenizer_path=TOKENIZER_PATH,
    embedding_layer_type=EMBEDDING_LAYER,
    model_name=MODEL_NAME,
    repetition=REPETITION,
    nn_error_weight=NN_ERROR_WEIGHT,
    label_smoothing=LABEL_SMOOTHING,
    learning_rate=LEARNING_RATE,
    max_loss=MAX_LOSS,
    retrain=RETRAIN,
    numchar=NUM_CHAR,
    toid=TO_ID,
    checkpoint_type=CHECKPOINT_TYPE,
    start_from_epoch=START_FROM_EPOCH,
    es_start_from_epoch=ES_START_FROM_EPOCH,
    use_early_stopping=USE_EARLY_STOPPING,
    patience=PATIENCE,
    monitor=MONITOR,
    mode=MODE,
    restore_best_weights=RESTORE_BEST_WEIGHTS,
    sklearn_models=SKLEARN_MODELS,
)

In [ ]:
# Run this to save model and tokenizer to Google Drive from Colab
from google.colab import auth
from googleapiclient.http import MediaFileUpload
from googleapiclient.discovery import build

auth.authenticate_user()
drive_service = build('drive', 'v3')

def save_file_to_drive(name, path):
    file_metadata = {'name': name, 'mimeType': 'application/octet-stream'}
    media = MediaFileUpload(path, mimetype='application/octet-stream', resumable=True)
    created = drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()
    print('File ID: {}'.format(created.get('id')))
    return created

save_file_to_drive('logseer.keras', MODEL_SAVE_PATH)
save_file_to_drive('tokenizer.pickle', TOKENIZER_PATH)